In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'Warehouse'))
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'Optimization'))

In [ ]:
import math
import random
from itertools import product

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from Aisle_Storage import Aisle
from Carton import Carton
from Warehouse_Builder import Warehouse_Builder, WarehouseConfig, AisleConfig
from Inventory_Builder import Inventory_Builder, InventoryConfig
from Inventory_Management import Inventory_Manager
from Affinity_Store import AffinityStore
from Workload_Builder import BatchConfig, Batch, Task
from Pick import PickConfig, PickSimulation
from Workload import WorkloadParams, aisle_workload

## Warehouse Configuration

72 aisles — every combination of handling type × category × unit type (24 combos × 3 aisles each).
All bins are `extra_large` so any Pallet tier fits without size-rank mismatches.

In [ ]:
# Reset class-level counters so IDs are deterministic when re-running
Aisle.next_aisle_id = 1
Carton.next_sku     = 1

HANDLING_TYPES   = ['conveyable', 'non-conveyable']
CATEGORY_TYPES   = ['food', 'clothing', 'electronic', 'furniture', 'seasonal', 'chemical']
UNIT_TYPES       = ['pallet', 'singleton']
AISLES_PER_COMBO = 3

all_combos   = list(product(HANDLING_TYPES, CATEGORY_TYPES, UNIT_TYPES))  # 24 combos
n_combos     = len(all_combos)
total_aisles = n_combos * AISLES_PER_COMBO   # 72

# Size each aisle so the warehouse starts at ~90% utilization after initial placement.
# Each SKU occupies exactly one bin (quantity=1), so total_bins ≈ num_skus / 0.90.
NUM_SKUS       = 200_000
TARGET_UTIL    = 0.90
bins_per_aisle = round(NUM_SKUS / TARGET_UTIL / total_aisles)
BAY_X          = round(math.sqrt(bins_per_aisle))
BAY_Y          = round(bins_per_aisle / BAY_X)

warehouse_config = WarehouseConfig(
    total_aisles  = total_aisles,
    aisle_splits  = [1 / n_combos] * n_combos,
    aisle_configs = [
        AisleConfig(
            handling_type  = h,
            storage_type   = c,
            unit_type      = u,
            bayXPerAisle   = BAY_X,
            bayYPerAisle   = BAY_Y,
            storage_sizes  = ['extra_large'],
        )
        for h, c, u in all_combos
    ],
)

warehouse = Warehouse_Builder().from_config(warehouse_config).build()
total_bins = len(warehouse.bins)
print(f'Aisles           : {len(warehouse.aisles)}')
print(f'Total bins       : {total_bins:,}  ({BAY_X}x{BAY_Y} per aisle)')
print(f'Target util      : {NUM_SKUS / total_bins:.1%}  ({NUM_SKUS:,} SKUs / {total_bins:,} bins)')
print(f'Combos           : {n_combos}  ({AISLES_PER_COMBO} aisles each)')

## Inventory

200 000 SKUs distributed evenly across all handling and category types.
Each SKU is placed with `quantity = 1` so every SKU occupies exactly one bin.

In [ ]:
random.seed(0)

inv_config = InventoryConfig(
    num_skus           = NUM_SKUS,
    handling_splits    = [0.5, 0.5],
    category_splits    = [1 / 6] * 6,
    singleton_fraction = 0.5,   # half the SKUs are sized for singleton forward-pick bins
)
inventory = Inventory_Builder().from_config(inv_config).build()
print(f'SKUs generated : {len(inventory.cartons):,}')

In [ ]:
manager = Inventory_Manager(warehouse)
manager.enqueue_all(inventory.cartons, quantity=1)
manager.summary()

In [ ]:
# AffinityStore indexes sku → lift_group once (O(N)).
# Pairwise lifts are generated lazily per batch and cached on disk, so the full
# O(N²/G) pre-computation cost is never paid.  Delete affinity_cache.db if you
# re-run with a different inventory seed.
store = AffinityStore('affinity_cache.db', seed=0)
store.index_inventory(inventory)
print(f'Affinity store ready  ({len(inventory.cartons):,} SKUs indexed)')

## Simulation — Event-Driven Batch Loop

25 pickers process batches sequentially.  A new batch is generated **only after
the last task of the current batch completes** — no batches overlap.  After each
batch `check_reorders()` is called so depleted SKUs are automatically restocked
before the next wave.

`AffinityStore` handles pairwise lift values.  Passing it to `Batch` triggers
random candidate selection (affinity-weighted selection over 100 k candidates
is infeasible), then loads within-group pairs for the **selected ~2 000 SKUs**
into a local dict — O(k²/G) per batch rather than O(N²/G) for the full
inventory.  The loaded dict is stored as `batch.aff` for analytics use.

In [ ]:
pick_cfg = PickConfig(
    num_pickers      = 25,
    x_move_time      = 1.0,
    y_move_time      = 0.5,
    pick_intercept   = 1.0,
    pick_weight_coef = 0.1,
    pick_volume_coef = 1e-4,
    cart_swap_coef   = 5.0,
)
wp = WorkloadParams.from_pick_config(pick_cfg)

batch_cfg = BatchConfig(
    inventory_size = len(inventory.cartons),
    mean_fraction  = 0.01,    # ~2 000 SKUs per batch
    std_fraction   = 0.002,
)
N_BATCHES = 100

print(f'Pickers          : {pick_cfg.num_pickers}')
print(f'Expected SKUs/batch : ~{round(batch_cfg.mean_fraction * batch_cfg.inventory_size):,}')

In [ ]:
random.seed(42)

cumulative_time = 0.0
batch_stats: list[dict] = []

for batch_idx in range(N_BATCHES):
    batch = Batch(batch_cfg, inventory, affinity=store)
    tasks = Task.from_batch(batch, warehouse)

    if not tasks:
        batch_stats.append({
            'batch'          : batch_idx,
            'skus_requested' : len(batch.items),
            'skus_picked'    : 0,
            'tasks'          : 0,
            'makespan'       : 0.0,
            'batch_start'    : cumulative_time,
            'batch_end'      : cumulative_time,
            'reorders'       : 0,
            'aff_pairs'      : len(batch.aff),
            'W_a_mean'       : 0.0,
            'W_a_total'      : 0.0,
        })
        continue

    # Compute W_a (aisle workload) for each task using PickConfig coefficients
    W_a_values = [
        aisle_workload(
            t.x_traversed,
            t.y_traversed,
            t.carts_required,
            [
                (
                    b.storage.carton.weight,
                    b.storage.carton.volume(),
                    t.items[b.storage.carton.sku],
                )
                for b in t.path
                if b.storage is not None and b.storage.carton.sku in t.items
            ],
            wp,
        )
        for t in tasks
    ]

    sim    = PickSimulation(tasks, pick_cfg)
    events = sim.run()

    # Makespan = time when the last picker finishes; next batch starts here
    makespan     = max(e.time for e in events if e.event_type == 'done')
    batch_start  = cumulative_time
    cumulative_time += makespan
    batch_end    = cumulative_time

    reorders = manager.check_reorders()

    batch_stats.append({
        'batch'          : batch_idx,
        'skus_requested' : len(batch.items),
        'skus_picked'    : sum(sum(t.items.values()) for t in tasks),
        'tasks'          : len(tasks),
        'makespan'       : round(makespan, 2),
        'batch_start'    : round(batch_start, 2),
        'batch_end'      : round(batch_end, 2),
        'reorders'       : len(reorders),
        'aff_pairs'      : len(batch.aff),
        'W_a_mean'       : round(sum(W_a_values) / len(W_a_values), 2),
        'W_a_total'      : round(sum(W_a_values), 2),
    })

df = pd.DataFrame(batch_stats)
print(f'Batches simulated : {len(df)}')
print(f'Total wall time   : {cumulative_time:,.1f} time units')
df.head(10)

## Results

In [ ]:
df[['skus_requested', 'tasks', 'makespan', 'reorders', 'W_a_mean', 'W_a_total']].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Makespan per batch
axes[0, 0].plot(df['batch'], df['makespan'], marker='.', ms=4, color='#2196F3')
axes[0, 0].set_title('Batch Makespan (25 pickers)')
axes[0, 0].set_xlabel('Batch index')
axes[0, 0].set_ylabel('Time units')

# SKUs picked per batch
axes[0, 1].bar(df['batch'], df['skus_requested'], width=0.8, color='#aec6cf')
axes[0, 1].set_title('SKUs per Batch')
axes[0, 1].set_xlabel('Batch index')
axes[0, 1].set_ylabel('SKU count')

# Cumulative throughput over wall time
axes[1, 0].plot(
    df['batch_end'],
    df['skus_requested'].cumsum(),
    marker='.', ms=4, color='#4CAF50',
)
axes[1, 0].set_title('Cumulative SKUs Picked vs Wall Time')
axes[1, 0].set_xlabel('Cumulative time units')
axes[1, 0].set_ylabel('Total SKUs picked')

# Reorder events
axes[1, 1].bar(df['batch'], df['reorders'], width=0.8, color='#ff6b6b')
axes[1, 1].set_title('Reorders Triggered per Batch')
axes[1, 1].set_xlabel('Batch index')
axes[1, 1].set_ylabel('SKUs reordered')

plt.tight_layout()
plt.show()

In [ ]:
# Batch timeline — each bar spans [batch_start, batch_end]; colour encodes task count
active = df[df['tasks'] > 0].copy()

norm  = plt.Normalize(active['tasks'].min(), active['tasks'].max())
cmap  = plt.cm.YlOrRd  # type: ignore[attr-defined]

fig, ax = plt.subplots(figsize=(14, 4))
for _, row in active.iterrows():
    ax.barh(
        0,
        row['batch_end'] - row['batch_start'],
        left=row['batch_start'],
        height=0.6,
        color=cmap(norm(row['tasks'])),
        edgecolor='none',
    )

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)  # type: ignore[attr-defined]
sm.set_array([])
plt.colorbar(sm, ax=ax, label='Tasks in batch')

ax.set_yticks([])
ax.set_xlabel('Cumulative time units')
ax.set_title('Batch Timeline — colour = number of aisle tasks', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of mean aisle workload W_a across batches
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df[df['W_a_mean'] > 0]['W_a_mean'], bins=30, edgecolor='black', color='#aec6cf')
ax.set_title('Distribution of Mean Aisle Workload W_a Across Batches')
ax.set_xlabel('W_a (time units)')
ax.set_ylabel('Batch count')
plt.tight_layout()
plt.show()